# PTI Whole-Body Human Generation — StyleGAN-Human + PTI + InsetGAN

End-to-end pipeline for **face-guided whole-body image generation** using
Pivotal Tuning Inversion (PTI) as the face encoder.

```
Input face photo
       │
       ▼
  PTI encoder  ──►  face latent codes (pivotal tuning, ~2 min/image)
       │
       ▼
  InsetGAN dual optimiser  ──►  face + body joint refinement
       │
       ▼
  Output: full-body PNG + MP4 video sequence
```

**vs ReStyle notebooks:** PTI fine-tunes the generator weights per image for
higher-fidelity face reconstruction, at the cost of longer inversion time
(~2 minutes vs ~10 seconds for ReStyle pSp).

**Before running:** Enable GPU via *Runtime → Change runtime type → GPU*


In [1]:
!nvidia-smi

Fri May 22 10:07:00 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   69C    P0             28W /   70W |    6285MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
# Patch custom_ops.py for PyTorch 2.x compatibility
import pathlib, re

# Find custom_ops.py wherever it lives in this runtime
matches = list(pathlib.Path('.').rglob('torch_utils/custom_ops.py'))
print("Found:", matches)

for path in matches:
    text = path.read_text()

    # Fix get_plugin: use load() return value, drop importlib.import_module
    text = re.sub(
        r'(torch\.utils\.cpp_extension\.load\(name=module_name, build_directory=build_dir,\s*'
        r'verbose=verbose_build, sources=digest_sources, \*\*build_kwargs\))\s*\n'
        r'(\s*)else:\s*\n'
        r'\s*torch\.utils\.cpp_extension\.load\(name=module_name, verbose=verbose_build, sources=sources, \*\*build_kwargs\)\s*\n'
        r'\s*module = importlib\.import_module\(module_name\)',
        lambda m: (
            'module = torch.utils.cpp_extension.load(name=module_name, build_directory=build_dir,\n'
            '                verbose=verbose_build, sources=digest_sources, **build_kwargs)\n'
            '        else:\n'
            '            module = torch.utils.cpp_extension.load(name=module_name, verbose=verbose_build, sources=sources, **build_kwargs)'
        ),
        text
    )

    # Fix get_plugin_v3: same pattern
    text = re.sub(
        r'torch\.utils\.cpp_extension\.load\(name=module_name, build_directory=cached_build_dir,\s*\n'
        r'\s*verbose=verbose_build, sources=cached_sources, \*\*build_kwargs\)\s*\n'
        r'(\s*)else:\s*\n'
        r'\s*torch\.utils\.cpp_extension\.load\(name=module_name, verbose=verbose_build, sources=sources, \*\*build_kwargs\)\s*\n'
        r'\s*# Load\.\s*\n'
        r'\s*module = importlib\.import_module\(module_name\)',
        (
            'module = torch.utils.cpp_extension.load(name=module_name, build_directory=cached_build_dir,\n'
            '                verbose=verbose_build, sources=cached_sources, **build_kwargs)\n'
            '        else:\n'
            '            module = torch.utils.cpp_extension.load(name=module_name, verbose=verbose_build, sources=sources, **build_kwargs)'
        ),
        text
    )

    path.write_text(text)
    print(f"Patched: {path}")


Found: [PosixPath('User-Whole-Body-Generation/StyleGAN_Human/torch_utils/custom_ops.py'), PosixPath('User-Whole-Body-Generation/PTI/torch_utils/custom_ops.py')]
Patched: User-Whole-Body-Generation/StyleGAN_Human/torch_utils/custom_ops.py
Patched: User-Whole-Body-Generation/PTI/torch_utils/custom_ops.py


In [3]:
import shutil, os
cache = os.path.expanduser("~/.cache/torch_extensions")
if os.path.exists(cache):
    shutil.rmtree(cache)
    print("Cleared torch_extensions cache")


Cleared torch_extensions cache


In [4]:
!pip install ninja -q


In [5]:
import torch
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("CUDA version:", torch.version.cuda)
!nvcc --version 2>/dev/null || echo "nvcc not found"


PyTorch: 2.10.0+cu128
CUDA available: True
CUDA version: 12.8
nvcc: NVIDIA (R) Cuda compiler driver
Copyright (c) 2005-2025 NVIDIA Corporation
Built on Fri_Feb_21_20:23:50_PST_2025
Cuda compilation tools, release 12.8, V12.8.93
Build cuda_12.8.r12.8/compiler.35583870_0


## 1. Environment Verification

Check that a GPU is attached. Enable GPU via *Runtime → Change runtime type → GPU*
if the output shows no CUDA devices.


In [6]:
!wget -q https://github.com/ninja-build/ninja/releases/download/v1.8.2/ninja-linux.zip
!sudo unzip -o ninja-linux.zip -d /usr/local/bin/
!sudo update-alternatives --install /usr/bin/ninja ninja /usr/local/bin/ninja 1 --force


Archive:  ninja-linux.zip
  inflating: /usr/local/bin/ninja    


The `-o` flag on `unzip` forces overwrite without prompting so re-running
this cell will not block on an interactive yes/no question.


# **PART 1 - User Whole body generation - Modified StyleGAN Human**

In [7]:
import os
if not os.path.exists('/content/User_Whole_Body_Generation'):
    !git clone https://github.com/Lakshmanaraja/User_Whole_Body_Generation.git
else:
    print('Repo already exists, skipping clone.')


Repo already exists, skipping clone.


# **Downloading Style Human Pre Trained Models**

In [8]:
import os

Content_base = '/content'
User_base = '/content/User_Whole_Body_Generation/'
Style_human_base = '/content/User_Whole_Body_Generation/StyleGAN_Human'
PTI_base =  '/content/User_Whole_Body_Generation/PTI' 

os.chdir(Style_human_base) #Setting the Current Working Directory

### Model Download Helper

`get_download_model_command()` builds a `wget` one-liner that handles
Google Drive's confirm-token redirect. Every model file is saved into
`<repo>/StyleGAN_Human/pretrained_models/` so all downstream code can
reference a single consistent path.


In [9]:
def get_download_model_command(file_id, file_name):
    """ Get wget download command for downloading the desired model and save to directory ../pretrained_models. """
    current_directory = os.getcwd()
    save_path = os.path.join(os.path.dirname(current_directory), f'{repo_name}',"pretrained_models")
    if not os.path.exists(save_path):
        os.makedirs(save_path)
    url = r"""wget --load-cookies /tmp/cookies.txt "https://docs.google.com/uc?export=download&confirm=$(wget --quiet --save-cookies /tmp/cookies.txt --keep-session-cookies --no-check-certificate 'https://docs.google.com/uc?export=download&id={FILE_ID}' -O- | sed -rn 's/.*confirm=([0-9A-Za-z_]+).*/\1\n/p')&id={FILE_ID}" -O {SAVE_PATH}/{FILE_NAME} && rm -rf /tmp/cookies.txt""".format(FILE_ID=file_id, FILE_NAME=file_name, SAVE_PATH=save_path)
    return url

### Download Body Generator Checkpoint

Downloads `stylegan2_1024.pkl` — the StyleGAN-Human whole-body generator
trained at 512×1024 portrait resolution. The `MODEL_PATHS` dict maps
experiment names to their Google Drive file IDs.


In [10]:
experiment_type = 'stylegan2_1024' 
repo_name = 'StyleGAN_Human'\
#version= 2
MODEL_PATHS = {
    "stylegan2_1024": {"id": "1FlAb1rYa0r_--Zj_ML8e6shmaF28hQb5", "name": "stylegan2_1024.pkl"},
} 
path = MODEL_PATHS[experiment_type]
download_command = get_download_model_command(file_id=path["id"], file_name=path["name"])
!{download_command}

--2026-05-22 10:07:10--  https://docs.google.com/uc?export=download&confirm=&id=1FlAb1rYa0r_--Zj_ML8e6shmaF28hQb5
Resolving docs.google.com (docs.google.com)... 192.178.163.113, 192.178.163.102, 192.178.163.138, ...
Connecting to docs.google.com (docs.google.com)|192.178.163.113|:443... connected.
HTTP request sent, awaiting response... 303 See Other
Location: https://drive.usercontent.google.com/download?id=1FlAb1rYa0r_--Zj_ML8e6shmaF28hQb5&export=download [following]
--2026-05-22 10:07:10--  https://drive.usercontent.google.com/download?id=1FlAb1rYa0r_--Zj_ML8e6shmaF28hQb5&export=download
Resolving drive.usercontent.google.com (drive.usercontent.google.com)... 173.194.203.132, 2607:f8b0:400e:c05::84
Connecting to drive.usercontent.google.com (drive.usercontent.google.com)|173.194.203.132|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 2437 (2.4K) [text/html]
Saving to: ‘/content/User-Whole-Body-Generation/StyleGAN_Human/pretrained_models/stylegan2_1024.pkl’


### Pretrained Models: FFHQ Generator & dlib

Downloads the FFHQ StyleGAN2 checkpoint (used by InsetGAN's face generator)
and two dlib data files for face detection and landmark prediction.


In [11]:
## Download pretrained StyleGAN on FFHQ 1024x1024 and dlib dat.
ffhq_ckpt = get_download_model_command(file_id="125OG7SMkXI-Kf2aqiwLLHyCvSW-gZk3M", file_name='ffhq.pkl')
dlib_detector = get_download_model_command(file_id="1MduBgju5KFNrQfDLoQXJ_1_h5MnctCIG", file_name='mmod_human_face_detector.dat')
dlib_landmark = get_download_model_command(file_id="1A82DnJBJzt8wI2J8ZrCK5fgHcQ2-tcWM", file_name='shape_predictor_68_face_landmarks.dat')
!{ffhq_ckpt}
!{dlib_detector}
!{dlib_landmark}

--2026-05-22 10:07:11--  https://docs.google.com/uc?export=download&confirm=&id=125OG7SMkXI-Kf2aqiwLLHyCvSW-gZk3M
Resolving docs.google.com (docs.google.com)... 192.178.163.113, 192.178.163.102, 192.178.163.138, ...
Connecting to docs.google.com (docs.google.com)|192.178.163.113|:443... connected.
HTTP request sent, awaiting response... 303 See Other
Location: https://drive.usercontent.google.com/download?id=125OG7SMkXI-Kf2aqiwLLHyCvSW-gZk3M&export=download [following]
--2026-05-22 10:07:11--  https://drive.usercontent.google.com/download?id=125OG7SMkXI-Kf2aqiwLLHyCvSW-gZk3M&export=download
Resolving drive.usercontent.google.com (drive.usercontent.google.com)... 173.194.203.132, 2607:f8b0:400e:c05::84
Connecting to drive.usercontent.google.com (drive.usercontent.google.com)|173.194.203.132|:443... connected.
HTTP request sent, awaiting response... 404 Not Found
2026-05-22 10:07:11 ERROR 404: Not Found.

--2026-05-22 10:07:12--  https://docs.google.com/uc?export=download&confirm=&id=1Md

### Additional Dependency: lpips

LPIPS (Learned Perceptual Image Patch Similarity) is the perceptual loss used
during the InsetGAN dual optimisation step.


In [12]:
!pip install lpips

# **PART 2 : PTI Encoder - Inversion**

# **Git and other Installation dependencies**

Current directory set to root folder to avoid any path issues

Downloading Restyle encoder from Git ( Utils folder is modified to accomadate  StyleGAN Human files - alignment.py and utils.py )

In [13]:
import os
import sys

os.chdir(User_base)

if not os.path.exists(PTI_base):
    !git clone https://github.com/danielroich/PTI.git
else:
    print('PTI repo already exists, skipping clone.')


PTI repo already exists, skipping clone.


Installing the dependencies

In [14]:
!pip install wandb

### PTI Directory & Core Imports

Changes into the PTI directory so that `from configs import ...` resolves
relative to the PTI package root, then imports all required modules.


In [15]:
os.chdir(PTI_base)

In [16]:
import pickle
import numpy as np
from PIL import Image
import torch
import torchvision.transforms as transforms
from configs import paths_config, hyperparameters, global_config
from utils.align_data import pre_process_images
from scripts.run_pti import run_PTI
from IPython.display import display
import matplotlib.pyplot as plt
from scripts.latent_editor_wrapper import LatentEditorWrapper

# Downloading Pretrained models

### Downloader & Pretrained Model Directories

`download_with_pydrive = False` — uses `gdown` so no Google OAuth
or `client_secrets.json` is needed.
To use PyDrive instead set it to `True` (requires Colab authentication).


In [17]:
import os

download_with_pydrive = False  # uses gdown — no OAuth required for public Drive files

class Downloader(object):
    def __init__(self, use_pydrive):
        self.use_pydrive = use_pydrive
        if self.use_pydrive:
            self.authenticate()

    def authenticate(self):
        from pydrive.auth import GoogleAuth
        from pydrive.drive import GoogleDrive
        from google.colab import auth
        from oauth2client.client import GoogleCredentials
        auth.authenticate_user()
        gauth = GoogleAuth()
        gauth.credentials = GoogleCredentials.get_application_default()
        self.drive = GoogleDrive(gauth)

    def download_file(self, file_id, file_dst):
        if self.use_pydrive:
            downloaded = self.drive.CreateFile({'id': file_id})
            downloaded.FetchMetadata(fetch_all=True)
            downloaded.GetContentFile(file_dst)
        else:
            os.system(f'gdown --id {file_id} -O "{file_dst}"')

downloader = Downloader(download_with_pydrive)
print('Downloader ready (gdown mode).')


Downloader ready (gdown mode).


Model downloading

In [18]:
current_directory = os.getcwd()
pretrained_models = os.path.join(os.path.dirname(current_directory), "PTI", "pretrained_models")
os.makedirs(pretrained_models, exist_ok=True)

In [19]:
import os, shutil

ffhq_dst = os.path.join(pretrained_models, 'ffhq.pkl')

if not os.path.exists(ffhq_dst):
    # Primary: try Google Drive (may return 404 on expired links)
    downloader.download_file("125OG7SMkXI-Kf2aqiwLLHyCvSW-gZk3M", ffhq_dst)

if not os.path.exists(ffhq_dst) or os.path.getsize(ffhq_dst) < 1_000_000:
    # Fallback 1: reuse the copy already downloaded in Part 1 (StyleGAN_Human)
    part1_ffhq = f'{Style_human_base}/pretrained_models/ffhq.pkl'
    if os.path.exists(part1_ffhq) and os.path.getsize(part1_ffhq) > 1_000_000:
        shutil.copy(part1_ffhq, ffhq_dst)
        print(f'Copied ffhq.pkl from StyleGAN_Human pretrained_models')
    else:
        # Fallback 2: NVIDIA CDN (StyleGAN2-ADA FFHQ 1024)
        print('Downloading ffhq.pkl from NVIDIA CDN...')
        os.system(f'wget -q "https://nvlabs-fi-cdn.nvidia.com/stylegan2-ada-pytorch/pretrained/ffhq.pkl" -O "{ffhq_dst}"')

if os.path.exists(ffhq_dst) and os.path.getsize(ffhq_dst) > 1_000_000:
    print(f'ffhq.pkl ready ({os.path.getsize(ffhq_dst) // 1_000_000} MB)')
else:
    print('WARNING: ffhq.pkl could not be obtained — PTI training will fail')


ffhq.pkl ready (381 MB)


In [20]:
## Download Dlib tool for alingment, used for preprocessing images before PTI
downloader.download_file("1xPmn19T6Bdd-_RfCVlgNBbfYoh1muYxR", os.path.join(pretrained_models, 'align.dat'))

In [22]:
# 1. Download ffhq.pkl directly from NVIDIA's CDN
!wget -q --show-progress \
    https://nvlabs-fi-cdn.nvidia.com/stylegan2-ada-pytorch/pretrained/ffhq.pkl \
    -O ./pretrained_models/ffhq.pkl

# 2. Re-download stylegan2_1024.pkl — bypass the virus scan warning
#    Replace FILE_ID with the actual ID from your project's download script
!pip install -q gdown --upgrade
!gdown "https://drive.google.com/uc?id=1FlAb1rYa0r_--Zj_ML8e6shmaF28hQb5&confirm=t" \
    -O ./pretrained_models/stylegan2_1024.pkl



./pretrained_models 100%[===================>] 363.94M  40.9MB/s    in 6.6s    
Downloading...
From (original): https://drive.google.com/uc?id=1FlAb1rYa0r_--Zj_ML8e6shmaF28hQb5
From (redirected): https://drive.google.com/uc?id=1FlAb1rYa0r_--Zj_ML8e6shmaF28hQb5&confirm=t&uuid=2d3b217b-92cb-4e24-8d23-c016af755194
To: /content/User-Whole-Body-Generation/PTI/pretrained_models/stylegan2_1024.pkl
100% 362M/362M [00:05<00:00, 68.9MB/s]


In [23]:
# Check what's actually in the pkl files
for pkl in ['./pretrained_models/ffhq.pkl', './pretrained_models/stylegan2_1024.pkl']:
    with open(pkl, 'rb') as f:
        header = f.read(200)
    print(pkl, '→', header[:80])

./pretrained_models/ffhq.pkl → b'\x80\x03}q\x00(X\x01\x00\x00\x00Gq\x01ctorch_utils.persistence\n_reconstruct_persistent_obj\nq\x02}q\x03(X\x04\x00\x00\x00ty'
./pretrained_models/stylegan2_1024.pkl → b'\x80\x03}q\x00(X\x13\x00\x00\x00training_set_kwargsq\x01}q\x02(X\n\x00\x00\x00class_nameq\x03X#\x00\x00\x00training.dataset.Image'


In [24]:
import os
for f in ['./pretrained_models/ffhq.pkl', './pretrained_models/stylegan2_1024.pkl']:
    print(f"{f}: {os.path.getsize(f)/1e6:.1f} MB")

./pretrained_models/ffhq.pkl: 381.6 MB
./pretrained_models/stylegan2_1024.pkl: 362.0 MB


### PTI Configuration & Paths

Sets all PTI config variables: input/output directories, model paths, and
hyperparameters. `use_locality_regularization = False` speeds up training at
the cost of some spatial fidelity.


In [25]:
image_dir_name = 'image'

use_image_online = False
use_multi_id_training = False
global_config.device = 'cuda'
paths_config.e4e = f'{PTI_base}/pretrained_models/e4e_ffhq_encode.pt'
paths_config.input_data_id = image_dir_name
paths_config.input_data_path = f'{PTI_base}/{image_dir_name}_processed'
paths_config.stylegan2_ada_ffhq = f'{PTI_base}/pretrained_models/ffhq.pkl'
paths_config.checkpoints_dir = PTI_base
paths_config.style_clip_pretrained_mappers = f'{PTI_base}/pretrained_models'
paths_config.dlib = f'{Style_human_base}/pretrained_models/shape_predictor_68_face_landmarks.dat'  # reuse file from Part 1
hyperparameters.use_locality_regularization = False


### create a folder for original image and image with aligned face

In [26]:
import pickle
import numpy as np
from PIL import Image
import torch
import torchvision.transforms as transforms
from configs import paths_config, hyperparameters, global_config
from utils.align_data import pre_process_images
from scripts.run_pti import run_PTI
from IPython.display import display
import matplotlib.pyplot as plt
from scripts.latent_editor_wrapper import LatentEditorWrapper

def get_latent_code_using_PTI (path_to_original_image_folder):

    #this is necessary for preprocessing the image,
    #it searches for images contained in the 'image_original' folder and preprocesses it,
    #and saves into 'image_preprocessed' folder.
    os.chdir(PTI_base)
    pre_process_images(f'{PTI_base}/image_original')

    # run PTI, latent codes will be saved to embeddings folder
    model_id = run_PTI(use_wandb=False, use_multi_id_training=use_multi_id_training)
    # this creates a folder containing the latent vector
    w_path_dir = f'{paths_config.embedding_base_dir}/{paths_config.input_data_id}'
    
    res = []
    for path in os.listdir(path_to_original_image_folder):
      filename, file_ext = os.path.splitext(path)
      if os.path.isfile(os.path.join(path_to_original_image_folder, path)) and file_ext == '.jpg':
        image_filename_short = os.path.basename(path).split('.')[0]
        #without the prior code working, nothing is received here.
        aligned_image = Image.open(f'{PTI_base}/image_processed/{image_filename_short}' + '.jpeg')
        aligned_image.resize((512,1024))  
        #path to folder containing the latent vector obtained from running the preceding line
        embedding_dir = f'{PTI_base}/embeddings/image/PTI/{image_filename_short}' 
        print(embedding_dir)
        print(w_path_dir)
        w_pivot = torch.load(f'{embedding_dir}/0.pt') #load the encoded image
        #saving encoding as a compressed npz file
        np.savez(f'w_latents.npz', w=w_pivot.cpu().detach().numpy())
        latent_code = np.load(f'{PTI_base}/w_latents.npz')
        print(latent_code['w'].shape)
        #the latent vector is stored as a dictionary with key 'w'
        res.append(latent_code['w'])
    return res    

# Loading the model in eval mode

Get Latent Code from a user image and passed to the inset GAN

### Working Directory for InsetGAN

Sets the working directory back to `StyleGAN_Human` so that InsetGAN's
relative imports (`insetgan1`, `legacy`, `utils/`) resolve correctly.
This `chdir` is kept separate from `get_latent_code_using_PTI()` above
so it only takes effect immediately before the InsetGAN import cell.


In [27]:
os.chdir(Style_human_base)

### Import Path Fix for InsetGAN

The repo was cloned as `User_Whole_Body_Generation` (underscores), which is already
a valid Python identifier — no symlink needed unlike repos with hyphens in the name.
However, `/content` must be on `sys.path` so Python can find the package root.


In [28]:
import os, sys

# /content must be on sys.path so that
# 'from User_Whole_Body_Generation.StyleGAN_Human.insetgan1 import InsetGAN' resolves.
if '/content' not in sys.path:
    sys.path.insert(0, '/content')
print('sys.path[0]:', sys.path[0])


sys.path[0]: /content


In [29]:
#This is part of Main function code in Insetgan implementation. This is
#modified such that face seed implemenation is removed
#Instead face_latent_code is used as input 

from User_Whole_Body_Generation.StyleGAN_Human.insetgan1 import InsetGAN
import torch
import torch.nn.functional as F
from tqdm import tqdm
from lpips import LPIPS
import numpy as np
from User_Whole_Body_Generation.StyleGAN_Human.torch_utils.models import Generator as bodyGAN
from User_Whole_Body_Generation.StyleGAN_Human.torch_utils.models_face import Generator as FaceGAN
import dlib
from User_Whole_Body_Generation.StyleGAN_Human.utils.alignment import align_face_for_insetgan
from User_Whole_Body_Generation.StyleGAN_Human.utils.util import visual,tensor_to_numpy, numpy_to_tensor
import legacy
import os
import click
import shutil


def user_body_optimisation( face_latent_codes , image_name ,  body_seed  , joint_optimization, joint_steps , trunc  ):

    os.chdir(Style_human_base)

    face_network = "./pretrained_models/ffhq.pkl"
    body_network = "./pretrained_models/stylegan2_1024.pkl"
  
    body_seed_int = int(body_seed)
    joint_steps_int = int(joint_steps)
    trunc_float = float(trunc)
    
    image_filename_short = image_name.split('.')[0]
    body_seed = body_seed_int
    joint_steps= joint_steps_int 
    truncation_psi = trunc_float 
    outdir = 'outputs/insetgan'
    video = 1

    file_name_png = f'./{outdir}/{image_filename_short}_{body_seed_int:04d}.png'
    file_name_mp4 = f'./{outdir}/{image_filename_short}_{body_seed_int:04d}.mp4' 
    directory_path = f'./{outdir}/{image_filename_short}_{body_seed:04d}'
    #Remove Directory functionality 
    if os.path.exists(directory_path) == True :
       print("removing folder")
       shutil.rmtree(directory_path)
    print(os.getcwd())
    if os.path.isfile(file_name_png):
       os.remove(file_name_png)
       print("deleting png")
    if os.path.isfile(file_name_mp4):
       print("deleting MP4")
       os.remove(file_name_mp4)
    

    device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
    insgan = InsetGAN(body_network, face_network)
    os.makedirs(outdir, exist_ok=True)
    face_mean = insgan.face_generator.mean_latent(3000)
    face_w =  torch.Tensor(face_latent_codes[0][4]).cuda().unsqueeze(0) #.unsqueeze(1)
    print(face_w.shape)
    face_w = truncation_psi * face_w + (1-truncation_psi) * face_mean
    face_img, _ = insgan.face_generator([face_w], input_is_latent=True)
  
    body_z = np.random.RandomState(body_seed).randn(1, 512).astype(np.float32)
    body_mean = insgan.body_generator.mean_latent(3000)
    body_w = insgan.body_generator.get_latent(torch.from_numpy(body_z).to(device))  # [N, L, C]
    body_w = truncation_psi * body_w + (1-truncation_psi) * body_mean
    body_img, _ = insgan.body_generator([body_w], input_is_latent=True)
    
      
    _, body_crop, _ = insgan.detect_face_dlib(body_img)
    face_img = F.interpolate(face_img, size=(body_crop[3]-body_crop[1], body_crop[2]-body_crop[0]), mode='area')
    cp_body = body_img.clone()
    cp_body[:, :, body_crop[1]:body_crop[3], body_crop[0]:body_crop[2]] = face_img
    
    optim_face_w, optim_body_w, crop = insgan.dual_optimizer(
        face_w, 
        body_w,
        joint_optimization,
        joint_steps=joint_steps,
        seed=f'{image_filename_short}_{body_seed:04d}',
        output_path=outdir,
        video=video
    )
    
    if video:
        ffmpeg_cmd = f"ffmpeg -hide_banner -loglevel error -i ./{outdir}/{image_filename_short}_{body_seed:04d}/%04d.jpg -c:v libx264 -vf fps=30 -pix_fmt yuv420p ./{outdir}/{image_filename_short}_{body_seed:04d}.mp4"
        os.system(ffmpeg_cmd)
  
    new_face_img, _ = insgan.face_generator([optim_face_w], input_is_latent=True)
    new_shape = crop[3] - crop[1], crop[2] - crop[0]
    new_face_img_crop = F.interpolate(new_face_img, size=new_shape, mode='area')
    seamless_body, _ = insgan.body_generator([optim_body_w], input_is_latent=True)
    seamless_body[:, :, crop[1]:crop[3], crop[0]:crop[2]] = new_face_img_crop
    temp = torch.cat([cp_body, seamless_body], dim=3)
    
    visual(temp, f"{outdir}/{image_filename_short}_{body_seed:04d}.png")
    
    path = f'{Style_human_base}/outputs/insetgan/'
    file_paths=['','']
    file_name_png = f'{image_filename_short}_{body_seed_int:04d}.png'
    file_paths[0] = os.path.join(path, file_name_png )
   
    file_paths[1] = os.path.join(path, file_name_mp4 )

    return (file_paths)   

### Run PTI Encoding & Face Inversion

Copies the source image into `image_original/`, aligns it with dlib's 68-point
predictor, then runs PTI to produce an optimised latent code saved in `embeddings/`.
PTI fine-tunes the generator per image — expect ~2 minutes on a T4 GPU.


In [30]:
import time
import shutil
# create folders for storing original and preprocessed images

os.chdir(PTI_base)
os.makedirs(f'./{image_dir_name}_original', exist_ok=True)
os.makedirs(f'./{image_dir_name}_processed', exist_ok=True)
os.chdir(f'./{image_dir_name}_original')

image_filename = 'Face2.jpg'

shutil.copy(f'{User_base}Face2.jpg' , f'{PTI_base}/image_original/{image_filename}')
    
path_to_original_image_folder = f'{PTI_base}/image_original'

sttime = time.time()

face_latent_codes = get_latent_code_using_PTI(path_to_original_image_folder)

100%|██████████| 1/1 [00:00<00:00,  1.77it/s]
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=AlexNet_Weights.IMAGENET1K_V1`. You can also use `weights=AlexNet_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Setting up [LPIPS] perceptual loss: trunk [alex], v[0.1], spatial [off]
Loading model from: /usr/local/lib/python3.12/dist-packages/lpips/weights/v0.1/alex.pth


  0%|          | 0/1 [00:00<?, ?it/s]

Setting up PyTorch plugin "bias_act_plugin"... Done.


Setting up PyTorch plugin "upfirdn2d_plugin"... Done.


/content/User-Whole-Body-Generation/PTI/training/projectors/w_projector.py:133: UserWarning: Converting a tensor with requires_grad=True to a scalar may lead to unexpected behavior.
Consider using tensor.detach() first. (Triggered internally at /pytorch/torch/csrc/autograd/generated/python_variable_methods.cpp:836.)
  logprint(f'step {step + 1:>4d}/{num_steps}: dist {dist:<4.2f} loss {float(loss):<5.2f}')
100%|██████████| 1/1 [03:01<00:00, 181.80s/it]


/content/User_Whole_Body_Generation/PTI/embeddings/image/PTI/Face2
./embeddings/image
(1, 18, 512)


In [31]:
import pathlib

alignment_path = pathlib.Path('/content/User_Whole_Body_Generation/StyleGAN_Human/utils/alignment.py')
text = alignment_path.read_text()
text = text.replace('PIL.Image.ANTIALIAS', 'PIL.Image.LANCZOS')
alignment_path.write_text(text)
print("Fixed")


Fixed


### Run Body Optimisation

Calls `user_body_optimisation()` with the PTI-encoded face latent codes.
`joint_optimization=False` skips the dual optimiser for faster results.
Adjust `body_seed`, `joint_steps`, and `trunc` to vary body appearance and quality.


In [32]:
file_paths = user_body_optimisation(face_latent_codes = face_latent_codes[0] ,image_name = image_filename, body_seed = 221, joint_optimization = False, joint_steps = 500, trunc = 0.6 )
endtime = time.time()
print(f"total time taken is {endtime-sttime}" )
print ( file_paths[1] )

/content/User-Whole-Body-Generation/StyleGAN_Human
Setting up [LPIPS] perceptual loss: trunk [alex], v[0.1], spatial [off]


/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=AlexNet_Weights.IMAGENET1K_V1`. You can also use `weights=AlexNet_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Loading model from: /usr/local/lib/python3.12/dist-packages/lpips/weights/v0.1/alex.pth
torch.Size([1, 512])


face: 24.0000, lr: 0.0006184665997806832, loss: 144.06, loss_coarse: 36.78;loss_border: 94.76, loss_face: 12.52;: 100%|██████████| 25/25 [00:07<00:00,  3.19it/s]
body: 149.0000, lr: 8.767851876239353e-05, loss: 28.12, loss_coarse: 18.03;loss_border: 10.08, loss_body: 3528.94, loss_reg: 0.01: 100%|██████████| 150/150 [00:43<00:00,  3.45it/s] 


total time taken is 255.48149919509888
/content/User_Whole_Body_Generation/StyleGAN_Human/outputs/insetgan/./outputs/insetgan/Face2_0221.mp4
